In [ ]:
repository_filter: list[str] = []
total_repositories_scanned: int = 0


In [ ]:
from moderne_visualizations_misc.reusable.data_loader import read_data_table
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.simplefilter("ignore")

# Primary data table (injected via NB_DATA_TABLE env var when run by the platform)
eol_df = read_data_table("../samples/end_of_life_dependencies.csv")
eol_df = qu.filter_repos(eol_df, repository_filter)
eol_df = qu.add_repo_short(eol_df)


In [ ]:
RED = "#EF5350"      # already end-of-life
AMBER = "#FFB74D"    # approaching end-of-life

if eol_df.empty:
    fig = qu.empty_figure("No end-of-life dependencies found for the selected filters.")
else:
    df = eol_df.copy()
    df["isEol"] = df["eol"].astype(str).str.strip().str.lower() == "true"

    # One row per distinct component occurrence per repository.
    comp = df.drop_duplicates(subset=["repoShort", "groupId", "artifactId", "version"])

    repos_affected = int(comp["repoShort"].nunique())
    already = int(comp["isEol"].sum())
    approaching = int((~comp["isEol"]).sum())
    products = int(comp["product"].nunique())

    # Panel A: repositories ranked by number of distinct EOL components.
    by_repo = comp.groupby(["repoShort", "isEol"]).size().unstack(fill_value=0)
    for col in (True, False):
        if col not in by_repo.columns:
            by_repo[col] = 0
    by_repo["total"] = by_repo[True] + by_repo[False]
    top_repo = by_repo.sort_values("total", ascending=True).tail(15)

    # Panel B: EOL products ranked by number of repositories that still use them.
    prod_repos = comp.groupby("product")["repoShort"].nunique().sort_values(ascending=True).tail(12)

    fig = make_subplots(
        rows=3,
        cols=4,
        specs=[
            [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
            [{"type": "bar", "colspan": 4}, None, None, None],
            [{"type": "bar", "colspan": 4}, None, None, None],
        ],
        row_heights=[0.20, 0.44, 0.36],
        vertical_spacing=0.16,
        subplot_titles=[
            "", "", "", "",
            "Repositories most exposed to end-of-life components"
            "<br><sup>distinct components per repository — already end-of-life vs approaching</sup>",
            "End-of-life products by number of repositories affected"
            "<br><sup>how many repositories still rely on each end-of-life product</sup>",
        ],
    )

    repos_suffix = f" / {total_repositories_scanned}" if total_repositories_scanned else ""
    title_font = {"size": 13, "color": "#555"}
    fig.add_trace(
        go.Indicator(mode="number", value=repos_affected, number={"suffix": repos_suffix},
                     title={"text": "Repos using<br>EOL components", "font": title_font}),
        row=1, col=1,
    )
    fig.add_trace(
        go.Indicator(mode="number", value=already, number={"font": {"color": RED}},
                     title={"text": "Components<br>already EOL", "font": title_font}),
        row=1, col=2,
    )
    fig.add_trace(
        go.Indicator(mode="number", value=approaching, number={"font": {"color": AMBER}},
                     title={"text": "Approaching<br>EOL", "font": title_font}),
        row=1, col=3,
    )
    fig.add_trace(
        go.Indicator(mode="number", value=products,
                     title={"text": "Distinct EOL<br>products", "font": title_font}),
        row=1, col=4,
    )

    eol_text = [str(v) if v else "" for v in top_repo[True]]
    soon_text = [str(v) if v else "" for v in top_repo[False]]
    fig.add_trace(
        go.Bar(y=list(top_repo.index), x=list(top_repo[True]), name="Already end-of-life",
               orientation="h", marker_color=RED, text=eol_text, textposition="inside",
               hovertemplate="%{y}: %{x} already-EOL components<extra></extra>"),
        row=2, col=1,
    )
    fig.add_trace(
        go.Bar(y=list(top_repo.index), x=list(top_repo[False]), name="Approaching end-of-life",
               orientation="h", marker_color=AMBER, text=soon_text, textposition="inside",
               hovertemplate="%{y}: %{x} approaching-EOL components<extra></extra>"),
        row=2, col=1,
    )

    fig.add_trace(
        go.Bar(y=list(prod_repos.index), x=list(prod_repos.values), orientation="h",
               marker={"color": list(prod_repos.values), "colorscale": "Reds"},
               text=list(prod_repos.values), textposition="outside", showlegend=False,
               hovertemplate="%{y}: used by %{x} repositories<extra></extra>"),
        row=3, col=1,
    )

    plural = "y" if repos_affected == 1 else "ies"
    share = f" ({repos_affected / total_repositories_scanned:.0%} of the portfolio)" if total_repositories_scanned else ""
    fig.update_layout(
        barmode="stack",
        height=900,
        template="plotly_white",
        title={"text": f"End-of-life exposure — {repos_affected} repositor{plural} rely on end-of-life components{share}",
               "x": 0.5, "xanchor": "center", "y": 0.975, "font": {"size": 17}},
        legend={"orientation": "h", "y": -0.07, "x": 0.5, "xanchor": "center"},
        margin={"t": 150, "l": 175, "r": 55, "b": 70},
    )

fig.show()
